In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [22]:
dataset = pd.read_csv('./cleanedDataset/cleanedDataset.csv' , index_col=False)
dataset.head()

,text,sentiment
0,قرأت إن في مشروع تطوير جديد بمنطقتنا,neutral
1,بدي أغير روتيني شوي، حاسس محتاج تغيير,neutral
2,سمعت إن في مطعم جديد افتتح بشارع الرينبو,neutral
3,الشغل هاي الأيام ضغط ما بيوصف، كتير تعبان,negative
4,والله زهقت من ناس بتحكي وما بتعمل شي,negative


In [23]:
x = dataset['text']
y = dataset['sentiment']

In [24]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [28]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

arabic_stopwords = [
    "من", "في", "على", "الى", "عن", "مع", "ان", "انها", "كان", "كانت", "هو", "هي", 
    "هذا", "هذه", "ذلك", "الذين", "التي", "كل", "قبل", "بعد", "كلما", "أو", "ام", "بل"
]

def normalize_arabic(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\u064B-\u0652]", "", text) 
    text = re.sub(r"[أإآ]", "ا", text)          
    text = re.sub(r"ة", "ه", text)               
    text = re.sub(r"ي", "ى", text)               
    return text


x_train_clean = x_train.apply(normalize_arabic)
x_test_clean = x_test.apply(normalize_arabic)


vectorizer = TfidfVectorizer(
    stop_words=arabic_stopwords, 
    ngram_range=(1, 2),
    max_features=5000  
)

x_train_tfidf = vectorizer.fit_transform(x_train_clean)
x_test_tfidf = vectorizer.transform(x_test_clean)

### Linear Classifier

In [29]:
from sklearn.linear_model import LogisticRegression
linear_model = LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced')
linear_model.fit(x_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [30]:
from sklearn.metrics import accuracy_score, classification_report
predictions = linear_model.predict(x_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

Accuracy: 67.40%

Classification Report:
              precision    recall  f1-score   support

    negative       0.63      0.69      0.66       410
     neutral       0.67      0.67      0.67       586
    positive       0.72      0.67      0.69       455

    accuracy                           0.67      1451
   macro avg       0.68      0.67      0.67      1451
weighted avg       0.68      0.67      0.67      1451



In [31]:
def predict_sentiment_using_linear_model(new_text):
    text_vector = vectorizer.transform([new_text])
    prediction = linear_model.predict(text_vector)
    return prediction[0]

In [32]:
sample_review = "بكره محمد"
print(f"Review: '{sample_review}' -> Predicted Sentiment: {predict_sentiment_using_linear_model(sample_review)}")

Review: 'بكره محمد' -> Predicted Sentiment: neutral


### Non-Linear Classifier

In [33]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
clf.fit(x_train_tfidf, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",500
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [34]:
clf_predictions = clf.predict(x_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, clf_predictions) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, clf_predictions))

Accuracy: 62.30%

Classification Report:
              precision    recall  f1-score   support

    negative       0.59      0.63      0.61       410
     neutral       0.62      0.62      0.62       586
    positive       0.65      0.62      0.64       455

    accuracy                           0.62      1451
   macro avg       0.62      0.62      0.62      1451
weighted avg       0.62      0.62      0.62      1451



### Transformer

In [35]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

x_train_dense = embedding_model.encode(x_train.tolist(), show_progress_bar=True)
x_test_dense = embedding_model.encode(x_test.tolist(), show_progress_bar=True)

transformer_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
)

transformer_classifier.fit(x_train_dense, y_train)

predictions = transformer_classifier.predict(x_test_dense)
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")
print(classification_report(y_test, predictions))

c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bilal\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 46/46 [00:05<00:00,  7.88i

Accuracy: 57.55%

              precision    recall  f1-score   support

    negative       0.52      0.62      0.57       410
     neutral       0.61      0.52      0.56       586
    positive       0.59      0.60      0.60       455

    accuracy                           0.58      1451
   macro avg       0.58      0.58      0.58      1451
weighted avg       0.58      0.58      0.58      1451



In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

label_dict = {'negative': 0, 'neutral': 1, 'positive': 2}
y_train_int = [label_dict[label] if isinstance(label, str) else label for label in y_train]
y_test_int = [label_dict[label] if isinstance(label, str) else label for label in y_test]

train_df = pd.DataFrame({'text': x_train, 'label': y_train_int})
test_df = pd.DataFrame({'text': x_test, 'label': y_test_int})


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


model_ckpt = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)


model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)


metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir="./arabic_dialect_transformer",
    learning_rate=2e-5,             
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,              
    weight_decay=0.01,
    eval_strategy="epoch",           
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(), 
    report_to="none"
)

# 7. Initialize Trainer using updated properties
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,     
    compute_metrics=compute_metrics,
)

# 8. Run Training Loop
print("Starting MARBERTv2 Fine-Tuning Process...")
trainer.train()

# 9. Print Final Evaluation 
print("\nFinal Performance on Validation Set:")
print(trainer.evaluate())

c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bilal\.cache\huggingface\hub\models--UBC-NLP--MARBERTv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26187.26it/s]
[transformers] BertForSe

Starting MARBERTv2 Fine-Tuning Process...


c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.545701,0.784287
2,0.656115,0.567939,0.785665
3,0.320800,0.630070,0.784976


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]
c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]
c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]



Final Performance on Validation Set:


c:\Users\bilal\Desktop\pattern recognetion project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.320800,0.545701,3,0.784287


{'eval_loss': 0.5457005500793457, 'eval_accuracy': 0.7842866988283942}


In [37]:
model.save_pretrained("./my_best_arabic_sentiment_model")
tokenizer.save_pretrained("./my_best_arabic_sentiment_model")

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.69s/it]


('./my_best_arabic_sentiment_model\\tokenizer_config.json',
 './my_best_arabic_sentiment_model\\tokenizer.json')